# 01 — Spatial Generalization: Select and cache regions

**Run this notebook once.** It downloads 2018–2022 NOAA observations for the ten tiles previously selected by the original seed-42 rejection run, verifies eligibility, and chooses one 96-hour window per drifter with seed 3001. It caches the **full selected region observations** and exact windows for repeatable experiments.

The cache is written to `data/external/spatial_generalization_cache/region_selection_seed3001.pkl.gz`. Rerunning this notebook can replace the cache, so do not rerun unless you intend to regenerate the dataset.


In [1]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from urllib.parse import urlencode, quote
from urllib.request import Request, urlopen
from io import BytesIO
import time, warnings
SEED=3001
rng=np.random.default_rng(SEED)
PROJECT_ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data'/'processed'/'region_A_observations.parquet').exists()),None)
if PROJECT_ROOT is None: raise FileNotFoundError('请从 data3001-project 或其子目录运行。')
BASE=PROJECT_ROOT/'data'/'processed'
A_PATH=BASE/'region_A_observations.parquet'
if A_PATH is None: raise FileNotFoundError('请设置 A_PATH 指向 Region A 观测数据。')
obs=pd.read_parquet(A_PATH) if A_PATH.suffix=='.parquet' else pd.read_csv(A_PATH,low_memory=False)
print('Region A:',A_PATH,obs.shape)
# Candidate sampling settings — choose BEFORE looking at model errors
TILE_DEGREES=10
# Reproducible recent-period window: limits download size while retaining a true out-of-region test.
TIME_START='2018-01-01T00:00:00Z'
TIME_END='2022-09-30T23:00:00Z'
MIN_DRIFTERS_PER_REGION=20
N_TEST_REGIONS=10
MAX_CANDIDATE_REQUESTS=10
REQUEST_PAUSE_SECONDS=0.25
NOAA='https://erddap.aoml.noaa.gov/gdp/erddap/tabledap/drifter_hourly_qc.csv'


Region A: /Users/roger/Documents/data3001-project/data/processed/region_A_observations.parquet (562944, 39)


## 1. Load and validate Region A; construct globally sampled test regions

NOAA `tabledap` constraints use `latitude`, `longitude`, and `time`. Only six required fields are downloaded. The ten fixed tiles are the first eligible tiles from the earlier seed-42 run; this efficient rerun does not repeat the expensive rejection search. A tile remains eligible only when it contains at least 20 distinct external drifters with a complete 96h window in 2018–2022. This is a reproducible observed-region sample, not a uniform probability sample of the global ocean.

In [2]:
from urllib.error import HTTPError
import ssl, certifi
SSL_CONTEXT=ssl.create_default_context(cafile=certifi.where())

def standardize_frame(df):
    aliases={'ID':['ID','id','drifter_id'],'time':['time','timestamp','datetime'],
             'latitude':['latitude','lat'],'longitude':['longitude','lon'],'ve':['ve'],'vn':['vn']}
    rename={}
    for dst,opts in aliases.items():
        col=next((x for x in opts if x in df.columns),None)
        if col is None: raise ValueError(f'Missing {dst}; available: {df.columns.tolist()}')
        rename[col]=dst
    z=df[list(rename)].rename(columns=rename).copy()
    z['ID']=z['ID'].astype(str)
    z['time']=pd.to_datetime(z.time,utc=True,errors='coerce')
    for c in ['latitude','longitude','ve','vn']:z[c]=pd.to_numeric(z[c],errors='coerce')
    z=z.dropna(subset=['ID','time','latitude','longitude','ve','vn'])
    z=z[z.latitude.between(-90,90) & z.longitude.between(-180,360)]
    z['longitude']=((z.longitude+180)%360)-180
    return z.sort_values(['ID','time']).drop_duplicates(['ID','time']).reset_index(drop=True)

obs=standardize_frame(obs)
# All IDs occurring in Region A are excluded from external test, including held-out A IDs.
a_ids=set(obs.ID)
print('Region A observations:',len(obs),'IDs:',len(a_ids))

def eligible_windows(frame):
    # Returns {ID: [candidate 96-point arrays]}, retaining windows wholly inside tile.
    out={}
    for did,g in frame.groupby('ID',sort=False):
        g=g.sort_values('time').reset_index(drop=True)
        dt=g.time.diff().dt.total_seconds().to_numpy()/3600
        breaks=np.r_[True,~np.isclose(dt[1:],1.)]
        run=np.cumsum(breaks)
        starts=[]
        for _,seg in g.groupby(run,sort=False):
            if len(seg)>=96:
                starts.extend(seg.index.to_numpy()[:len(seg)-95].tolist())
        if starts:out[did]=starts
    return out

def fetch_tile(lat0,lon0):
    lat1=min(lat0+TILE_DEGREES,90);lon1=min(lon0+TILE_DEGREES,180)
    variables='ID,time,latitude,longitude,ve,vn'
    constraints=(f'&time>={TIME_START}&time<={TIME_END}'
                 f'&latitude>={lat0}&latitude<={lat1}'
                 f'&longitude>={lon0}&longitude<={lon1}')
    from urllib.parse import quote

    url = NOAA + '?' + variables + quote(
        constraints,
        safe='&=:-.'
    )

    request = Request(
        url,
        headers={"User-Agent": "DATA3001-academic-research/1.0"}
    )

    try:
        with urlopen(request, timeout=120, context=SSL_CONTEXT) as resp:
            payload = resp.read()

    except HTTPError as e:
        body = e.read().decode("utf-8", errors="replace")

        if e.code == 404 and "nRows = 0" in body:
            return pd.DataFrame(
                columns=["ID", "time", "latitude", "longitude", "ve", "vn"]
            )

        print("Request URL:", url)
        print("HTTP status:", e.code)
        print("NOAA response:", body[:1500])
        raise
    z=pd.read_csv(BytesIO(payload),low_memory=False)
    z=standardize_frame(z)
    return z[(z.latitude>=lat0)&(z.latitude<lat1)&(z.longitude>=lon0)&(z.longitude<lon1)]

# Sample disjoint grid tiles; avoid tile(s) intersecting Region A observation footprint.
# IMPORTANT: Region A may not be a simple rectangle; conservative exclusion uses every
# occupied 10° tile in its observations (not merely one mean coordinate).
a_tiles=set(zip(np.floor(obs.latitude/TILE_DEGREES).astype(int),
                np.floor(obs.longitude/TILE_DEGREES).astype(int)))
# These are the first ten eligible tiles selected by the original seed-42 rejection run.
# Fixing them here avoids downloading and discarding up to 100 very large candidate tiles.
all_tiles=[(20,-180),(40,-70),(30,-60),(30,-80),(-60,170),(-20,0),(-20,100),(20,140),(-60,-130),(-50,-100)]
region_rng = np.random.default_rng(42)

region_frames={};region_rows=[];failed=[]
for attempt, (lat0, lon0) in enumerate(
        all_tiles[:MAX_CANDIDATE_REQUESTS], 1
    ):
        if len(region_frames) >= N_TEST_REGIONS:
            break

        print(
            f"Checking region {attempt}/{MAX_CANDIDATE_REQUESTS}: "
            f"lat={lat0}, lon={lon0}",
            flush=True
        )

        try:
            tile=fetch_tile(lat0,lon0)
            tile=tile[~tile.ID.isin(a_ids)]
            valid=eligible_windows(tile)
            if len(valid)<MIN_DRIFTERS_PER_REGION:
                region_rows.append((lat0,lon0,len(tile),len(valid),'ineligible'))
                continue
            name=f'R{len(region_frames)+1:02d}'
            region_frames[name]=tile[tile.ID.isin(valid)].copy()
            region_rows.append((lat0,lon0,len(tile),len(valid),name))
            print(f'{name}: latitude [{lat0},{lat0+10}), longitude [{lon0},{lon0+10}), valid drifters={len(valid)}')
        except Exception as ex:
            failed.append((lat0, lon0, str(ex)[:180]))

            print(
                f"Request failed for ({lat0}, {lon0}): {ex}"
            )

            continue
        time.sleep(REQUEST_PAUSE_SECONDS)
region_selection=pd.DataFrame(region_rows,columns=['lat0','lon0','rows','valid_drifters','status'])
display(region_selection[region_selection.status.str.startswith('R')])
print('Selected regions:',len(region_frames),'failed requests:',len(failed))
if len(region_frames)<N_TEST_REGIONS:
    raise RuntimeError(f'Only {len(region_frames)} eligible tiles found. Increase MAX_CANDIDATE_REQUESTS, expand TIME_START/END, or lower MIN_DRIFTERS_PER_REGION, then rerun from the top.')


Region A observations: 562944 IDs: 521
Checking region 1/10: lat=20, lon=-180


/var/folders/1c/wbxz8krj5nn0086j5f4mk5hc0000gn/T/ipykernel_47129/427439396.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  z['time']=pd.to_datetime(z.time,utc=True,errors='coerce')


R01: latitude [20,30), longitude [-180,-170), valid drifters=78


Checking region 2/10: lat=40, lon=-70


/var/folders/1c/wbxz8krj5nn0086j5f4mk5hc0000gn/T/ipykernel_47129/427439396.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  z['time']=pd.to_datetime(z.time,utc=True,errors='coerce')


R02: latitude [40,50), longitude [-70,-60), valid drifters=37


... (22 outputs left out)

/var/folders/1c/wbxz8krj5nn0086j5f4mk5hc0000gn/T/ipykernel_47129/427439396.py:15: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  z['time']=pd.to_datetime(z.time,utc=True,errors='coerce')


R10: latitude [-50,-40), longitude [-100,-90), valid drifters=54


,lat0,lon0,rows,valid_drifters,status
0,20,-180,188656,78,R01
1,40,-70,49685,37,R02
2,30,-60,491877,174,R03
3,30,-80,202303,125,R04
4,-60,170,129150,116,R05
5,-20,0,25159,20,R06
6,-20,100,119046,60,R07
7,20,140,241452,95,R08
8,-60,-130,85399,98,R09
9,-50,-100,77562,54,R10


Selected regions: 10 failed requests: 0


## 2. ID-disjoint Region A train/validation and one random window per drifter

Test IDs are excluded from A by construction. A is split by ID into training and validation only; all hyperparameter choices use validation A. If a drifter occurs in two sampled test tiles, assign it to the first selected tile so that every test drifter contributes at most one case overall.

In [3]:
# A: split by ID; then sample one valid window per drifter.
# Reset RNG on every run so ID splits and sampled windows are reproducible.
rng = np.random.default_rng(SEED)
a_valid=eligible_windows(obs)
a_eligible=obs[obs.ID.isin(a_valid)].copy()
a_id_list=np.array(sorted(a_valid));rng.shuffle(a_id_list)
cut=int(.8*len(a_id_list))
if cut<5 or cut==len(a_id_list):raise ValueError('Not enough eligible Region A drifters for training/validation')
id_sets={'train':set(a_id_list[:cut]),'validation':set(a_id_list[cut:])}
selected=[];selection_log=[]
def add_windows(frame,partition,region,exclude=None):
    valid=eligible_windows(frame)
    for did,starts in valid.items():
        if exclude is not None and did in exclude:continue
        g=frame[frame.ID==did].sort_values('time').reset_index(drop=True)
        start=int(rng.choice(starts))
        w=g.iloc[start:start+96].copy()
        w['forecast_case_id']=f'{region}_{did}'
        w['window_hour']=np.arange(96)
        w['partition']=partition
        w['region']=region
        selected.append(w)
        selection_log.append((did,region,partition,len(starts),start))
for part in ['train','validation']:
    add_windows(a_eligible[a_eligible.ID.isin(id_sets[part])],part,'A')
used_test_ids=set()
for region,tile in region_frames.items():
    before=len(selected)
    add_windows(tile,'test',region,exclude=used_test_ids)
    used_test_ids.update(w.ID.iloc[0] for w in selected[before:])
chosen=pd.concat(selected,ignore_index=True)
selection_log=pd.DataFrame(selection_log,columns=['ID','region','partition','candidate_windows','selected_start'])
assert chosen.groupby('ID').partition.nunique().max()==1
assert not (set(chosen[chosen.partition=='test'].ID)&a_ids)
display(selection_log.groupby(['region','partition']).agg(n_drifters=('ID','nunique'),candidate_windows=('candidate_windows','sum')))


n_drifters  candidate_windows
region partition                                
A      train              358             348430
       validation          90              94865
R01    test                78             166239
R02    test                37              40189
R03    test               161             412256
R04    test                62              75717
R05    test               116             110572
R06    test                20              22025
R07    test                59             102079
R08    test                84             192485
R09    test                51              40174
R10    test                18              18528

## 3. Save selected regions and cases to disk

In [4]:
# Save the exact selected test regions and windows for reproducible model experiments.
# This cell is the ONLY place where region data is cached; the modeling notebook never calls NOAA.
from pathlib import Path
import pickle, gzip, json
CACHE_DIR = PROJECT_ROOT / 'data' / 'external' / 'spatial_generalization_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / 'region_selection_seed3001.pkl.gz'
cache_payload = {
    'obs': obs, 'region_frames': region_frames, 'region_selection': region_selection,
    'chosen': chosen, 'selection_log': selection_log,
    'id_sets': id_sets, 'a_ids': a_ids,
    'seed': SEED, 'region_seed': 42,
    'settings': dict(TILE_DEGREES=TILE_DEGREES, TIME_START=TIME_START,
                     TIME_END=TIME_END, MIN_DRIFTERS_PER_REGION=MIN_DRIFTERS_PER_REGION,
                     N_TEST_REGIONS=N_TEST_REGIONS),
}
with gzip.open(CACHE_FILE, 'wb', compresslevel=3) as f:
    pickle.dump(cache_payload, f, protocol=pickle.HIGHEST_PROTOCOL)
print('Saved region selection cache:', CACHE_FILE)
print('Test regions:', sorted(region_frames))
print('Selected 96-hour cases by region:')
display(selection_log.groupby(['region','partition']).size().rename('cases').to_frame())


Saved region selection cache: /Users/roger/Documents/data3001-project/data/external/spatial_generalization_cache/region_selection_seed3001.pkl.gz
Test regions: ['R01', 'R02', 'R03', 'R04', 'R05', 'R06', 'R07', 'R08', 'R09', 'R10']
Selected 96-hour cases by region:


cases
region partition        
A      train         358
       validation     90
R01    test           78
R02    test           37
R03    test          161
R04    test           62
R05    test          116
R06    test           20
R07    test           59
R08    test           84
R09    test           51
R10    test           18